# M04D: Self-Consistency & Output Quality

One response can be wrong. Generate multiple, pick the best.

**Topics:**
- Majority vote consensus picker
- Quality scorer
- Self-consistency validator

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


def truncate_response(text, max_length=1200):
    """Truncate text for readability."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"


def normalize_label(text):
    """Normalize a one-word label."""
    text = text.strip().lower()
    return text.strip(" .,!?:;\"'")


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🎯 What is Self-Consistency?

Generate multiple responses, compare them, pick the best. Consistent answers = higher confidence. Errors get caught through redundancy.

**Example:**
```
Prompt: "Is this email urgent? Answer: low, medium, or high"
Response 1: "medium"
Response 2: "medium"
Response 3: "medium"
Consensus: medium (high confidence!)
```

**Important:** This measures model agreement, not ground truth. Models can be consistently wrong.

---

## 📊 When to Use Self-Consistency

**✅ Use it when:**
- High-stakes decisions (medical, legal, financial)
- Factual accuracy critical (math, data analysis)
- Classification edge cases (spam, content moderation)

**❌ Skip it when:**
- Creative tasks — diversity is the goal
- Simple queries — one response is enough
- Cost/speed constraints — N responses = N× cost

**Golden rule:** Use to verify or select — not to increase creativity.

---

## 🗳️ Technique #1: Majority Vote (Consensus)

Generate N responses, count which answer appears most, return the majority. Best for discrete answers (categories, yes/no, A/B/C).

In [ ]:
print("🗳️  MAJORITY VOTE TECHNIQUE")
print("="*60)

email = """The dashboard is loading slowly for some users. 
A few customers mentioned it today."""

print(f"Email: {email}\n")
print("Generating 5 classifications...\n")

prompt = f"""Classify this email urgency as: low, medium, or high

Email: "{email}"

Answer with just one word: low, medium, or high"""

responses = []
for i in range(5):
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        instructions="Answer with exactly one word: low, medium, or high."
    )
    # Note: assumes one-word response; production code should validate against expected labels
    answer = normalize_label(response.output_text)
    responses.append(answer)
    print(f"Response {i+1}: {answer}")

print("="*60)

### 📊 Analyze the Votes

In [ ]:
vote_counts = Counter(responses)
top_answer, top_count = vote_counts.most_common(1)[0]

consensus = top_answer
confidence = top_count / len(responses)

print("\n" + "="*60)
print("📊 VOTE RESULTS")
print("="*60)

for answer, count in vote_counts.most_common():
    percentage = (count / len(responses)) * 100
    print(f"{answer}: {count} votes ({percentage:.0f}%)")

print(f"\n✅ Consensus: {consensus}")
print(f"📈 Confidence: {confidence * 100:.0f}%")

### 🔑 Why This Works

High consensus (5/5) = trust the answer. Low consensus = ambiguous task or unclear prompt.

---

## 🔢 Technique #2: Numerical Consensus

Voting works for numbers too, but needs normalization first — "14622.13" and "14,622.133" won't match as strings.

In [ ]:
print("🔢 NUMERICAL CONSENSUS")
print("="*60)

math_problem = "What is 17.3% of 84,521?"

print(f"Problem: {math_problem}\n")
print("Generating 5 calculations...\n")

calculations = []
for i in range(5):
    response = client.responses.create(
        model=MODEL,
        input=math_problem,
        instructions="Return only the number. No symbols, units, or text."
    )
    raw = response.output_text.strip().replace(",", "")
    try:
        result = round(float(raw), 2)
        calculations.append(result)
        print(f"Response {i+1}: {result}")
    except ValueError:
        print(f"Response {i+1}: {raw} (skipped - not a number)")

if not calculations:
    print("\n❌ No valid numbers returned. Try a clearer prompt.")
else:
    # Find consensus
    vote_counts = Counter(calculations)
    top_answer, top_count = vote_counts.most_common(1)[0]

    consensus = top_answer
    confidence = top_count / len(calculations)

    print(f"\n✅ Consensus answer: {consensus}")
    print(f"📈 Confidence: {confidence*100:.0f}%")
print("="*60)

---

## ⭐ Technique #3: Quality Scoring

Voting fails when every response is unique. Instead: generate N, score each, return the best. Best for explanations, summaries, creative writing.

In [ ]:
print("⭐ QUALITY SCORING TECHNIQUE")
print("="*60)

question = "Explain how email gets delivered from sender to recipient."

print(f"Question: {question}\n")
print("Generating 3 explanations...\n")

explanations = []
for i in range(3):
    response = client.responses.create(
        model=MODEL,
        input=question,
        instructions="Explain in simple terms. Be concise (2-3 sentences)."
    )
    explanation = response.output_text.strip()
    explanations.append(explanation)
    print(f"Response {i+1}:")
    print(truncate_response(explanation, max_length=300))
    print()

print("="*60)

### ⭐ Score Each Response

In [ ]:
print("\n📊 SCORING RESPONSES")
print("="*60)

scores = []
for i, explanation in enumerate(explanations, 1):
    scoring_prompt = f"""Rate this explanation on a scale of 1-10:

Criteria:
- Accuracy (technically correct)
- Simplicity (easy to understand)
- Completeness (covers key points)

Explanation: "{explanation}"

Return a single number (1-10) with no other text."""
    
    score_response = client.responses.create(
        model=MODEL,
        input=scoring_prompt,
        instructions="Return only a number between 1 and 10. No words, no explanation."
    )
    
    try:
        score = float(score_response.output_text.strip())
        scores.append(score)
        print(f"Response {i}: Score = {score}/10")
    except ValueError:
        scores.append(0)
        print(f"Response {i}: Score = N/A (parsing error)")

print("="*60)

### 🏆 Select the Best

In [ ]:
best_index = scores.index(max(scores))
best_explanation = explanations[best_index]
best_score = scores[best_index]

print("\n" + "="*60)
print("🏆 BEST RESPONSE")
print("="*60)
print(f"Score: {best_score}/10")
print(f"\nExplanation:\n{best_explanation}")
print("="*60)

### 🔑 Quality Scoring vs. Voting

Voting works when answers repeat (classification, math).  

Scoring works when every response is unique (explanations, summaries).

### 📋 When to Use Each Technique

<div style="text-align: left; display: inline-block;">

| Technique | Best For | Tradeoff |
|-----------|----------|----------|
| Majority Vote | Classification, yes/no, multiple choice | N calls + normalization |
| Numerical Consensus | Math, data extraction | N calls + float conversion |
| Quality Scoring | Open-ended responses, explanations | 2N calls |

</div>

---

## 💰 Cost vs Quality Tradeoff

**Worth it:** High-stakes decisions where cost of error > API cost.

**Skip it:** High-volume or low-stakes tasks.

---

### 💪 Your Turn: Build a Fact-Checker

Use majority vote to classify claims as true/false/uncertain with confidence scores.

**Note:** This checks model consistency, not truth. Real fact-checking requires external sources.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Build a Fact-Checker
# --------------------------------------------------------------
# Objective: Use self-consistency to verify claims with confidence scores.
# Hint: Use normalize_label() to clean up model responses before counting
# Hint: Counter from setup cell gives you .most_common()

def check_fact(claim, n_responses=5):
    """Check if a claim is true using self-consistency."""
    # TODO: Generate n_responses for this claim
    # Hint: Prompt the model to answer ONLY 'true', 'false', or 'uncertain'
    
    # TODO: Count votes using Counter
    
    # TODO: Calculate confidence percentage
    
    # TODO: Return dict with verdict, confidence, all_responses
    pass


# --------------------------------------------------------------
test_claims = [
    "The Earth orbits the Sun.",
    "Python was created in 1991.",
    "The human brain has 100 trillion neurons."
]

# TODO: Loop through test_claims and print verdict + confidence for each

print("🔍 Build your fact-checker above!")

---

## 🎯 Key Takeaways

**🗳️ Majority Vote for Discrete Answers (such as positive/negative/neutral):**
- Normalize responses before counting
- Pick most common answer as verdict
- High agreement = high confidence, low agreement = ambiguous prompt

**🔢 Numerical Consensus for Math and Data:**
- Convert to float and round before comparing
- Handles formatting differences ("14,622" vs "14622")
- Same voting logic, just with number normalization

**⭐ Quality Scoring for Open-Ended Responses:**
- Generate N responses → Score each → Pick highest
- Costs 2N API calls (generate + score)
- Use for explanations, summaries, and creative tasks

**💰 When Self-Consistency Is Worth the Cost:**
- High-stakes decisions where cost of error > API cost
- Skip for simple queries or creative tasks
- **The Flow:** Choose technique → Generate N → Normalize → Apply consensus/scoring → Return with confidence

---

### 📍 Next Step

**M04E: Advanced Reasoning Lab** — Build a multi-turn research assistant that combines all the techniques from Module 4.

---

## 🔧 Troubleshooting

**Responses not agreeing?**
- Prompt might be ambiguous — add clearer constraints
- Task may be inherently subjective
- Try adding "Answer with exactly one word" instruction

**Consensus too low?**
- Normal for subjective or ambiguous tasks
- Try quality scoring instead of voting
- Consider if self-consistency is right for this task

**Quality scores unreliable?**
- Scorer criteria might be too vague
- Add specific dimensions (clarity, accuracy, completeness)
- Verify scorer prompt is unambiguous

**Too expensive?**
- Reduce N (try 3 instead of 5)
- Only use for critical decisions
- Cache results for repeated inputs

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---